# Figuring out how to get `ffmpeg`, `torch` and `torchcodec` to play nicely together

In [ ]:
# user configurable settings
path_to_ffmpeg = "ffmpeg"


In [ ]:
import log_config # to override default and use loguru instead
log_config.setup_logging()
from loguru import logger

# a bit of a hack to get windows to find the DLLs for FFmpeg
# Use Python's Windows DLL API (3.8+). Add the folder that holds avcodec/avformat/avutil DLLs.
# TorchCodec README + version matrix: https://github.com/pytorch/torchcodec  (docs)
# Torchaudio FFmpeg install notes on Windows: https://docs.pytorch.org/audio/main/installation.html  (install tips)

from pathlib import Path
import os, sys

ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"
assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir
os.add_dll_directory(str(ffmpeg_dll_dir))  # Python 3.8+ DLL search

import torch, torchcodec, platform, subprocess
print("exe:", sys.executable)
print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
subprocess.run(["ffmpeg", "-version"], check=True) # do I need to add the ffmpeg dir to the PATH?


# yields error:
# ---------------------------------------------------------------------------
# OSError                                   Traceback (most recent call last)
# Cell In[3], line 11
#       9 ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"
#      10 assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir
# ---> 11 os.add_dll_directory(str(ffmpeg_dll_dir))  # Python 3.8+ DLL search
#      13 import torch, torchcodec, platform, subprocess
#      14 print("exe:", sys.executable)

# File <frozen os>:1172, in add_dll_directory(path)
#    1162 """Add a path to the DLL search path.
#    1163 
#    1164 This search path is used when resolving dependencies for imported
#    (...)   1169 using it in a with statement.
#    1170 """
#    1171 import nt
# -> 1172 cookie = nt._add_dll_directory(path)
#    1173 return _AddedDllDirectory(
#    1174     path,
#    1175     cookie,
#    1176     nt._remove_dll_directory
#    1177 )

# OSError: [WinError 87] The parameter is incorrect: 'ffmpeg\\bin'


OSError: [WinError 87] The parameter is incorrect: 'ffmpeg\\bin'

So it would appear that I'm sending the wrong path to the DLL function. Let's try that again after a small tweak... Reading some documentation, it would appear that the path should be _absolute_ and not relative. So let's try that...


In [ ]:
# a bit of a hack to get windows to find the DLLs for FFmpeg
# Use Python's Windows DLL API (3.8+). Add the folder that holds avcodec/avformat/avutil DLLs.
# TorchCodec README + version matrix: https://github.com/pytorch/torchcodec  (docs)
# Torchaudio FFmpeg install notes on Windows: https://docs.pytorch.org/audio/main/installation.html  (install tips)

from pathlib import Path
import os, sys

ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"
# assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir
print(f"{ffmpeg_dll_dir.exists()=}")
print(f"{ffmpeg_dll_dir=}")
print(f"{str(ffmpeg_dll_dir)= }")
print(f"{os.path.abspath(str(ffmpeg_dll_dir))=}")
abs_dll_path = os.path.abspath(str(ffmpeg_dll_dir))
os.add_dll_directory(abs_dll_path)  # Python 3.8+ DLL search

import torch, torchcodec, platform, subprocess

print("exe:", sys.executable)
print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
subprocess.run(["ffmpeg", "-version"], check=True)  # do I need to add the ffmpeg dir to the PATH?

# this go around produces output:
# ffmpeg_dll_dir.exists()=True
# ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
# str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
# os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
# exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
# torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2

# but also errors out with:

# ---------------------------------------------------------------------------
# FileNotFoundError                         Traceback (most recent call last)
# Cell In[11], line 22
#      20 print("exe:", sys.executable)
#      21 print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
# ---> 22 subprocess.run(["ffmpeg", "-version"], check=True)  # do I need to add the ffmpeg dir to the PATH?

# File ~\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\subprocess.py:554, in run(input, capture_output, timeout, check, *popenargs, **kwargs)
#     551     kwargs['stdout'] = PIPE
#     552     kwargs['stderr'] = PIPE
# --> 554 with Popen(*popenargs, **kwargs) as process:
#     555     try:
#     556         stdout, stderr = process.communicate(input, timeout=timeout)

# File ~\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\subprocess.py:1038, in Popen.__init__(self, args, bufsize, executable, stdin, stdout, stderr, preexec_fn, close_fds, shell, cwd, env, universal_newlines, startupinfo, creationflags, restore_signals, start_new_session, pass_fds, user, group, extra_groups, encoding, errors, text, umask, pipesize, process_group)
#    1034         if self.text_mode:
#    1035             self.stderr = io.TextIOWrapper(self.stderr,
#    1036                     encoding=encoding, errors=errors)
# -> 1038     self._execute_child(args, executable, preexec_fn, close_fds,
#    1039                         pass_fds, cwd, env,
#    1040                         startupinfo, creationflags, shell,
#    1041                         p2cread, p2cwrite,
#    1042                         c2pread, c2pwrite,
#    1043                         errread, errwrite,
#    1044                         restore_signals,
#    1045                         gid, gids, uid, umask,
#    1046                         start_new_session, process_group)
#    1047 except:
#    1048     # Cleanup if the child failed starting.
#    1049     for f in filter(None, (self.stdin, self.stdout, self.stderr)):

# File ~\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\subprocess.py:1552, in Popen._execute_child(self, args, executable, preexec_fn, close_fds, pass_fds, cwd, env, startupinfo, creationflags, shell, p2cread, p2cwrite, c2pread, c2pwrite, errread, errwrite, unused_restore_signals, unused_gid, unused_gids, unused_uid, unused_umask, unused_start_new_session, unused_process_group)
#    1550 # Start the process
#    1551 try:
# -> 1552     hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
#    1553                              # no special security
#    1554                              None, None,
#    1555                              int(not close_fds),
#    1556                              creationflags,
#    1557                              env,
#    1558                              cwd,
#    1559                              startupinfo)
#    1560 finally:
#    1561     # Child is launched. Close the parent's copy of those pipe
#    1562     # handles that only the child should have open.  You need
#    (...)   1565     # pipe will not close when the child process exits and the
#    1566     # ReadFile will hang.
#    1567     self._close_pipe_fds(p2cread, p2cwrite,
#    1568                          c2pread, c2pwrite,
#    1569                          errread, errwrite)

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\debugpy\_vendored\pydevd\_pydev_bundle\pydev_monkey.py:911, in create_CreateProcess.<locals>.new_CreateProcess(app_name, cmd_line, *args)
#     908     cmd_line = patch_arg_str_win(cmd_line)
#     909     send_process_created_message()
# --> 911 return getattr(_subprocess, original_name)(app_name, cmd_line, *args)

# FileNotFoundError: [WinError 2] The system cannot find the file specified


ffmpeg_dll_dir.exists()=True
ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2


FileNotFoundError: [WinError 2] The system cannot find the file specified

This appears to point me towards the issue of the code not being able to find the `ffmpeg` executable, which, as far as I understand, could be using the sys.path OR NOT, depending on python version. Let's see if I can just call `ffmpeg/ffmpeg` or `ffmpeg/ffmpeg.exe` directly. I also added the `shell = True` parameter to the `subprocess.run()` call, but that didn't entirely fix it. Reading more, it's ill advised, but using `shell=True` does seem to be getting me closer to a solution...

In [ ]:
# a bit of a hack to get windows to find the DLLs for FFmpeg
# Use Python's Windows DLL API (3.8+). Add the folder that holds avcodec/avformat/avutil DLLs.
# TorchCodec README + version matrix: https://github.com/pytorch/torchcodec  (docs)
# Torchaudio FFmpeg install notes on Windows: https://docs.pytorch.org/audio/main/installation.html  (install tips)

from pathlib import Path
import os, sys

ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"
# assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir
print(f"{ffmpeg_dll_dir.exists()=}")
print(f"{ffmpeg_dll_dir=}")
print(f"{str(ffmpeg_dll_dir)= }")
print(f"{os.path.abspath(str(ffmpeg_dll_dir))=}")
abs_dll_path = os.path.abspath(str(ffmpeg_dll_dir))
os.add_dll_directory(abs_dll_path)  # Python 3.8+ DLL search

import torch, torchcodec, platform, subprocess

print("exe:", sys.executable)
print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
subprocess.run(["ffmpeg/ffmpeg", "-version"], check=True, shell=True)

# we're getting closer.... Output is:
# ffmpeg_dll_dir.exists()=True
# ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
# str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
# os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
# exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
# torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2

# and then it errors out with:
# ---------------------------------------------------------------------------
# CalledProcessError                        Traceback (most recent call last)
# Cell In[16], line 22
#      20 print("exe:", sys.executable)
#      21 print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
# ---> 22 subprocess.run(["ffmpeg/ffmpeg", "-version"], check=True, shell=True) 

# File ~\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\subprocess.py:577, in run(input, capture_output, timeout, check, *popenargs, **kwargs)
#     575     retcode = process.poll()
#     576     if check and retcode:
# --> 577         raise CalledProcessError(retcode, process.args,
#     578                                  output=stdout, stderr=stderr)
#     579 return CompletedProcess(process.args, retcode, stdout, stderr)
# 
# CalledProcessError: Command '['ffmpeg/ffmpeg', '-version']' returned non-zero exit status 1.


ffmpeg_dll_dir.exists()=True
ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2


CalledProcessError: Command '['ffmpeg/ffmpeg', '-version']' returned non-zero exit status 1.

Well, it appears that it has found the ffmpeg executable now, so that's good. But apparently when it runs `ffmpeg -version`, it returns a non-zero exit code, which I'll need to investigate... From reading documentation, it appears the `check` flag passed to subprocess.run() is intended to raise an exception if the command returns a non-zero exit code (as it's currently doing), so let's turn that flag off and retry.

In [ ]:
# a bit of a hack to get windows to find the DLLs for FFmpeg
# Use Python's Windows DLL API (3.8+). Add the folder that holds avcodec/avformat/avutil DLLs.
# TorchCodec README + version matrix: https://github.com/pytorch/torchcodec  (docs)
# Torchaudio FFmpeg install notes on Windows: https://docs.pytorch.org/audio/main/installation.html  (install tips)

from pathlib import Path
import os, sys

ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"
assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir
print(f"{ffmpeg_dll_dir.exists()=}")
print(f"{ffmpeg_dll_dir=}")
print(f"{str(ffmpeg_dll_dir)= }")
print(f"{os.path.abspath(str(ffmpeg_dll_dir))=}")
abs_dll_path = os.path.abspath(str(ffmpeg_dll_dir))
os.add_dll_directory(abs_dll_path)  # Python 3.8+ DLL search

import torch, torchcodec, platform, subprocess

print("exe:", sys.executable)
print("torch", torch.__version__, "torchcodec", torchcodec.__version__, "py", platform.python_version())
result = subprocess.run(["ffmpeg/ffmpeg", "-version"], check=False, shell=True)
print(result)
print(result.returncode)
print(result.args)
print(result.stdout)
print(result.stderr)

# output:
# ffmpeg_dll_dir.exists()=True
# ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
# str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
# os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
# exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
# torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2
# CompletedProcess(args=['ffmpeg/ffmpeg', '-version'], returncode=1)
# 1
# ['ffmpeg/ffmpeg', '-version']
# None
# None


ffmpeg_dll_dir.exists()=True
ffmpeg_dll_dir=WindowsPath('ffmpeg/bin')
str(ffmpeg_dll_dir)= 'ffmpeg\\bin'
os.path.abspath(str(ffmpeg_dll_dir))='z:\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin'
exe: z:\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe
torch 2.10.0+cpu torchcodec 0.10.0 py 3.14.2
CompletedProcess(args=['ffmpeg/ffmpeg', '-version'], returncode=1)
1
['ffmpeg/ffmpeg', '-version']
None
None


Victory! No more nasty errors! Hurray! Now, we should have `ffmpeg` working, as well as our DLL directory for `torch` and `torchcodec`. Now, where were we...